<a href="https://colab.research.google.com/github/DwiAyuniRohana/data-science-2025/blob/main/Pertemuan3_DwiAyuniRohana_250401020173.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Nama:** Dwi Ayuni Rohana

**NIM:** 250401020173

**Kelas:** IF405

**Prodi:** Informatika

In [ ]:
# ==============================================================================
# PIPELINE DATA CLEANING — HOUSING DATASET
# Mata Kuliah: Pengantar Data Science
# ==============================================================================

import pandas as pd
import numpy as np
import os
import warnings

# Mengabaikan warning kosmetik agar output bersih
warnings.filterwarnings('ignore')

# --- DETEKSI AUTOMATIS DATASET ---
# Jika file 'housing_dirty.csv' belum ada, kita buatkan data sampel otomatis agar tidak error saat di-run
if not os.path.exists('housing_dirty.csv'):
    np.random.seed(42)
    n = 130
    sampel_id = [f"REG-{i:03d}" for i in range(1, n + 1)]
    sampel_luas = np.random.normal(120, 35, n)
    sampel_harga = sampel_luas * 12 + np.random.normal(0, 50, n)
    sampel_kota = np.random.choice([' Jakarta ', 'Surabaya', ' bandung '], n)
    sampel_kamar = np.random.choice([2, 3, 4, 5], n).astype(float)
    sampel_tahun = np.random.randint(2000, 2024, n)
    sampel_kondisi = np.random.choice([' Bagus ', 'butuh renovasi', 'Baru'], n)

    # Menyisipkan missing value buatan agar pipeline pembersihan berjalan
    sampel_luas[np.random.choice(n, 18, replace=False)] = np.nan
    sampel_harga[np.random.choice(n, 17, replace=False)] = np.nan
    sampel_kamar[np.random.choice(n, 10, replace=False)] = np.nan

    # Menyisipkan outlier ekstrem buatan
    sampel_harga[5] = 4500.0
    sampel_luas[12] = 850.0

    df_raw = pd.DataFrame({
        'id': sampel_id, 'luas_m2': sampel_luas, 'harga_juta': sampel_harga,
        'kota': sampel_kota, 'kamar': sampel_kamar, 'tahun_bangun': sampel_tahun, 'kondisi': sampel_kondisi
    })
    df_raw.to_csv('housing_dirty.csv', index=False)
    print("👉 Berhasil membuat file contoh 'housing_dirty.csv' secara otomatis.\n")

# ==============================================================================
# PROSES PIPELINE EKSEKUSI UTAMA
# ==============================================================================
# STEP 0 — Load & eksplorasi awal
df = pd.read_csv('housing_dirty.csv')
print('=== LANGKAH 0: INSPEKSI AWAL ===')
print('Shape awal:', df.shape)
print('\nJumlah Missing Values Sebelum Cleaning:')
print(df.isnull().sum())
print("-" * 50)

# STEP 1 — Hapus Duplikat
df.drop_duplicates(inplace=True)
print('\n=== LANGKAH 1: REMOVAL DUPLIKAT ===')
print('Setelah hapus duplikat:', df.shape)
print("-" * 50)

# STEP 2 — Normalisasi String
df['kota'] = df['kota'].str.strip().str.title()
df['kondisi'] = df['kondisi'].str.strip().str.lower()

# STEP 3 — Imputasi Missing Values
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

# STEP 4 — Tangani Outlier (IQR Fence)
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    df[col] = df[col].clip(Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)

# STEP 5 — Validasi & Ekspor
assert df.isnull().sum().sum() == 0, 'Masih ada missing!'
assert df.duplicated().sum() == 0, 'Masih ada duplikat!'
print('\n=== LANGKAH 5: VALIDASI AKHIR ===')
print('Shape akhir dataset bersih:', df.shape)
df.to_csv('housing_clean.csv', index=False)
print('📢 Sukses: Dataset bersih berhasil divalidasi dan disimpan dengan aman!')

=== LANGKAH 0: INSPEKSI AWAL ===
Shape awal: (130, 7)

Jumlah Missing Values Sebelum Cleaning:
id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64
--------------------------------------------------

=== LANGKAH 1: REMOVAL DUPLIKAT ===
Setelah hapus duplikat: (130, 7)
--------------------------------------------------

=== LANGKAH 5: VALIDASI AKHIR ===
Shape akhir dataset bersih: (130, 7)
📢 Sukses: Dataset bersih berhasil divalidasi dan disimpan dengan aman!


# Kesimpulan

Pada praktikum ini saya mempelajari proses **Data Cleaning** sebagai salah satu tahapan penting dalam Data Science. Kegiatan yang dilakukan meliputi inspeksi data awal, penghapusan data duplikat, normalisasi data kategorikal, penanganan missing values menggunakan metode imputasi, deteksi dan penanganan outlier dengan metode Interquartile Range (IQR), serta validasi hasil pembersihan data sebelum data disimpan kembali ke dalam file baru.

Temuan utama dari praktikum ini menunjukkan bahwa kualitas data dapat ditingkatkan secara signifikan melalui proses pembersihan yang sistematis. Missing values pada kolom numerik berhasil diatasi menggunakan nilai median, sedangkan kolom kategorikal menggunakan modus. Selain itu, data kategorikal yang tidak konsisten berhasil dinormalisasi, dan nilai-nilai outlier ekstrem dapat dikendalikan menggunakan metode clipping berdasarkan batas IQR. Setelah seluruh tahapan dilakukan, dataset menjadi lebih bersih, konsisten, dan siap digunakan untuk analisis maupun pembangunan model Machine Learning.

Keterbatasan dari praktikum ini adalah proses cleaning masih menggunakan aturan sederhana dan dataset yang digunakan bersifat simulasi. Pada kasus dunia nyata, proses pembersihan data sering kali lebih kompleks karena melibatkan data yang lebih besar, berbagai jenis kesalahan input, serta outlier yang tidak selalu dapat ditangani dengan satu metode saja. Pertanyaan yang muncul adalah bagaimana memilih metode imputasi dan penanganan outlier yang paling tepat untuk berbagai karakteristik dataset yang berbeda.
